# 00 · Briefing room
*"I recruit based on potential." — Nick Fury*

Before Fury hands you the keys, he wants to know one thing: can you get into the building? This notebook is a smoke test.
It proves that your workspace can run code, that you know who and where you are, and that the notebook can see this repo's code.
It changes nothing in your workspace.

| Concept | Where you meet it |
|---|---|
| Notebook, compute and the `spark` session | the next section, Step 1 |
| Identity and Unity Catalog's three-level names | Step 2 |
| Git folder: importing the repo's own code | Step 3 |

Deep dive: [docs/00-briefing-room.md](../../docs/00-briefing-room.md)

**Needs:** an Azure Databricks workspace and this repo cloned there as a **Git folder** (Workspace → Create → Git folder). Any compute works; **serverless** is the simplest.
No earlier chapters, no tables, no special permissions. The **Expected** values below are expectations until you have run the notebook in your own workspace.

## What "running on Databricks" means

**The idea:** a notebook is only a document in your browser. When you run a cell, the code is sent to **compute** (serverless, or a cluster you created), which executes it and sends the result back.
Databricks has already created a Spark session on that compute and put it in a variable called `spark`. `display(...)` is a second ready-made helper that renders a table in the notebook. Neither needs an `import`.

```
  browser (notebook)  ── cell code ──▶  compute (serverless or cluster)  ──▶  Spark  ──▶  data in cloud storage
        ▲                                        │
        └──────────── display(...) result ───────┘
```

**How this notebook implements it**

* Step 1 builds a tiny DataFrame with `spark.range(5)`: no files, no tables, just proof that Spark answers.
* Step 2 asks Spark SQL who you are and which catalog is your default. Unity Catalog names every table with three parts, `catalog.schema.table` (for example `shield.bronze.agents` later in this course). The catalog is the top level, so the current catalog is where an unqualified table name would land.
* Step 3 imports `shield_data` from the repo's `src/` folder. That works because a **Git folder** is a real clone of the repo inside the workspace, so the notebook sits next to the code it needs.

**Why it helps:** if one of these three fails, every later chapter fails the same way, and it is much easier to find out here.

**Not covered yet:** creating the catalog and schemas (chapter 01), choosing between serverless and classic compute (chapter 02), Delta tables (chapter 04).

## Setup

In [ ]:
def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    rows = [(what, fmt(got), fmt(want), "✅" if got == want else "❌") for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))

## 1 · Prove the workshop works

### Step 1 · Can Spark run?
**Goal:** prove the notebook is attached to working compute, and that you can see a result.

| Input | Value |
|---|---|
| `spark.range(5)` | five rows with a column `id`, renamed to `agent_number` |

**Expected:** a table with one column, `agent_number`, and five rows (0 to 4). The first run on serverless can take a little longer while compute starts.

What you will see:

| agent_number |
|---|
| 0 |
| 1 |
| 2 |
| 3 |
| 4 |

Safe to re-run: nothing is stored.

In [ ]:
agents_demo = spark.range(5).withColumnRenamed("id", "agent_number")
display(agents_demo)

show_checks([
    ("rows", agents_demo.count(), 5),
    ("columns", agents_demo.columns, ["agent_number"]),
])

### Step 2 · Which identity and catalog am I using?
**Goal:** see who Databricks thinks you are and which catalog is your default, so you recognise it when permissions come up in chapter 01.

**Expected:** one row. `me` is the account you signed in with, `catalog` and `schema` are your workspace's defaults. These names differ per workspace, so there is no single right answer and the check table only confirms that each value came back.
The catalog `shield` does not exist yet: chapter 01 creates it.

Safe to re-run.

In [ ]:
whoami = spark.sql("SELECT current_user() AS me, current_catalog() AS catalog, current_schema() AS schema")
display(whoami)

me = whoami.first()
show_checks([
    ("current_user returned", bool(me["me"]), True),
    ("current_catalog returned", bool(me["catalog"]), True),
    ("current_schema returned", bool(me["schema"]), True),
])

### Step 3 · Can I import the repo's code?
**Goal:** prove the notebook can import `shield_data`, the repo's synthetic-data package, from `src/`. Chapters 02, 03, 06 and others rely on it.

**Expected:** one line, `10 S.H.I.E.L.D. locations loaded from <path>`, where `<path>` is your Git folder in the workspace. The check table confirms 10 locations and that the module was loaded from this repo's `src/` folder.

If you see `ModuleNotFoundError: No module named 'shield_data'`, the notebook is probably not inside a Git folder (for example it was uploaded as a single file), so `../../src` does not exist next to it. Clone the repo as a Git folder and open the notebook from there.

Re-running adds the same path to `sys.path` again, which is harmless.

In [ ]:
import os
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(repo_root, "src"))

import shield_data
from shield_data.generate import make_locations

shield_locations = make_locations()
print(f"{len(shield_locations)} S.H.I.E.L.D. locations loaded from {repo_root}")

show_checks([
    ("locations", len(shield_locations), 10),
    ("module loaded from this repo's src", os.path.abspath(shield_data.__file__).startswith(os.path.join(repo_root, "src")), True),
])

## 👀 In the workspace

* **Compute selector** (top right of the notebook): shows what the notebook ran on. This is the answer to "where did my code run?".
* **Git branch button** next to the notebook name: opens the Git dialog of the Git folder (pull, switch branch, commit).
* **Workspace** in the sidebar: your Git folder shows as a folder with a Git icon, with `notebooks/`, `src/` and `docs/` inside.

## 🧹 Clean up

Nothing to delete: this notebook creates no tables, files or Azure resources.

## 🎯 Recap

| Check | What it proved |
|---|---|
| Step 1 | The notebook is attached to compute and `spark` answers. |
| Step 2 | You know your user and your default catalog. |
| Step 3 | The repo's `src/` code is importable from the Git folder. |

Cost reminder: compute and Event Hubs cost money. Prefer serverless, stop what you are not using, and run `azd down --purge` when you pause.

Try-it-yourself mini-projects: [docs/00-briefing-room.md](../../docs/00-briefing-room.md#try-it-yourself)

> 💡 Next: chapter 01 builds the platform: the Unity Catalog external location, the `shield` catalog, its schemas and volumes.